In [2]:

print("Ok")


Ok


In [3]:

%pwd

'c:\\projects\\medibot\\Elarova\\research'

In [4]:
import os
os.chdir("../")

In [5]:
%pwd

'c:\\projects\\medibot\\Elarova'

In [6]:

from langchain.document_loaders import PyPDFLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [12]:
#Extract Data From the PDF File
def load_pdf_file(data):
    loader= DirectoryLoader(data,
                            glob="*.pdf",
                            loader_cls=PyPDFLoader)

    documents=loader.load()

    return documents

In [13]:
extracted_data=load_pdf_file(data='Data/')

In [15]:
# extracted_data

In [16]:
#Split the Data into Text Chunks
def text_split(extracted_data):
    text_splitter=RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=20)
    text_chunks=text_splitter.split_documents(extracted_data)
    return text_chunks

In [17]:
text_chunks=text_split(extracted_data)
print("Length of Text Chunks", len(text_chunks))

Length of Text Chunks 5860


In [19]:
# text_chunks

In [20]:

from langchain.embeddings import HuggingFaceEmbeddings

In [21]:

#Download the Embeddings from Hugging Face
def download_hugging_face_embeddings():
    embeddings=HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
    return embeddings

In [22]:

embeddings = download_hugging_face_embeddings()

C:\Users\LENOVO\AppData\Local\Temp\ipykernel_5572\2503581564.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings=HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')


In [23]:

query_result = embeddings.embed_query("Hello world")
print("Length", len(query_result))

Length 384


In [25]:
# query_result

In [56]:

from dotenv import load_dotenv
load_dotenv()

True

In [57]:
PINECONE_API_KEY=os.environ.get('PINECONE_API_KEY')
OPENAI_API_KEY=os.environ.get('OPENAI_API_KEY')

In [58]:
from pinecone import Pinecone, ServerlessSpec
import os

# Retrieve Pinecone API key (replace "PINECONE_API_KEY" with your actual API key)
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
if not PINECONE_API_KEY:
    raise ValueError("PINECONE_API_KEY environment variable is not set.")

# Initialize Pinecone client
pc = Pinecone(api_key=PINECONE_API_KEY)

# Define index parameters
index_name = "elarova"
dimension = 384
metric = "cosine"
cloud = "aws"
region = "us-east-1"

# Check if the index already exists
if index_name in pc.list_indexes():
    print(f"Index '{index_name}' already exists.")
else:
    try:
        # Create the index
        pc.create_index(
            name=index_name,
            dimension=dimension,
            metric=metric,
            spec=ServerlessSpec(
                cloud=cloud,
                region=region
            )
        )
        print(f"Index '{index_name}' created successfully.")
    except Exception as e:
        print(f"Failed to create index: {e}")

Failed to create index: (409)
Reason: Conflict
HTTP response headers: HTTPHeaderDict({'content-type': 'text/plain; charset=utf-8', 'access-control-allow-origin': '*', 'vary': 'origin,access-control-request-method,access-control-request-headers', 'access-control-expose-headers': '*', 'x-pinecone-api-version': '2024-07', 'x-cloud-trace-context': '8e3c01bad1bb5390b8683f38f4845b30', 'date': 'Wed, 21 May 2025 12:23:07 GMT', 'server': 'Google Frontend', 'Content-Length': '85', 'Via': '1.1 google', 'Alt-Svc': 'h3=":443"; ma=2592000,h3-29=":443"; ma=2592000'})
HTTP response body: {"error":{"code":"ALREADY_EXISTS","message":"Resource  already exists"},"status":409}



In [60]:
import os

os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY


In [61]:
# Embed each chunk and upsert the embeddings into your Pinecone index.
from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore.from_documents(
    documents=text_chunks,
    index_name=index_name,
    embedding=embeddings, 
)

In [62]:
# Load Existing index 

from langchain_pinecone import PineconeVectorStore
# Embed each chunk and upsert the embeddings into your Pinecone index.
docsearch = PineconeVectorStore.from_existing_index(
    index_name=index_name,
    embedding=embeddings
)

In [63]:
docsearch

In [64]:
retriever = docsearch.as_retriever(search_type="similarity", search_kwargs={"k":3})

In [65]:
retrieved_docs = retriever.invoke("What is Acoustic neuroma?")

In [66]:

retrieved_docs

[Document(id='437e1ac1-4126-47a4-83e6-41201890353d', metadata={'creationdate': '2004-12-18T17:00:02-05:00', 'creator': 'PyPDF', 'moddate': '2004-12-18T16:15:31-06:00', 'page': 42.0, 'page_label': '43', 'producer': 'PDFlib+PDI 5.0.0 (SunOS)', 'source': 'Data\\Medical_book.pdf', 'total_pages': 637.0}, page_content='GALE ENCYCLOPEDIA OF MEDICINE 2 29\nAcoustic neuroma\nGEM - 0001 to 0432 - A  10/22/03 1:41 PM  Page 29'),
 Document(id='9578c8a1-425e-4a51-be8f-d703175809fe', metadata={'creationdate': '2004-12-18T17:00:02-05:00', 'creator': 'PyPDF', 'moddate': '2004-12-18T16:15:31-06:00', 'page': 42.0, 'page_label': '43', 'producer': 'PDFlib+PDI 5.0.0 (SunOS)', 'source': 'Data\\Medical_book.pdf', 'total_pages': 637.0}, page_content='GALE ENCYCLOPEDIA OF MEDICINE 2 29\nAcoustic neuroma\nGEM - 0001 to 0432 - A  10/22/03 1:41 PM  Page 29'),
 Document(id='9188ee39-ab1e-4086-bd90-a86a4bac3398', metadata={'creationdate': '2004-12-18T17:00:02-05:00', 'creator': 'PyPDF', 'moddate': '2004-12-18T16:15

In [2]:
!pip install langchain-openai==0.0.8


  Using cached langchain_openai-0.0.8-py3-none-any.whl.metadata (2.5 kB)
  Using cached langsmith-0.1.147-py3-none-any.whl.metadata (14 kB)
  Using cached packaging-23.2-py3-none-any.whl.metadata (3.2 kB)
  Using cached tenacity-8.5.0-py3-none-any.whl.metadata (1.2 kB)
Using cached langchain_openai-0.0.8-py3-none-any.whl (32 kB)
Using cached langsmith-0.1.147-py3-none-any.whl (311 kB)
Using cached packaging-23.2-py3-none-any.whl (53 kB)
Using cached tenacity-8.5.0-py3-none-any.whl (28 kB)
  Attempting uninstall: tenacity
    Found existing installation: tenacity 9.0.0
    Uninstalling tenacity-9.0.0:
      Successfully uninstalled tenacity-9.0.0
  Attempting uninstall: packaging
    Found existing installation: packaging 24.2
    Uninstalling packaging-24.2:
      Successfully uninstalled packaging-24.2
  Attempting uninstall: langsmith
    Found existing installation: langsmith 0.3.13
    Uninstalling langsmith-0.3.13:
      Successfully uninstalled langsmith-0.3.13


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-community 0.3.19 requires langchain<1.0.0,>=0.3.20, which is not installed.
langchain-community 0.3.19 requires langchain-core<1.0.0,>=0.3.41, but you have langchain-core 0.1.53 which is incompatible.
langchain-huggingface 0.1.2 requires langchain-core<0.4.0,>=0.3.15, but you have langchain-core 0.1.53 which is incompatible.
langchain-pinecone 0.2.3 requires langchain-core<1.0.0,>=0.3.34, but you have langchain-core 0.1.53 which is incompatible.
langchain-tests 0.3.14 requires langchain-core<1.0.0,>=0.3.43, but you have langchain-core 0.1.53 which is incompatible.
langchain-text-splitters 0.3.8 requires langchain-core<1.0.0,>=0.3.51, but you have langchain-core 0.1.53 which is incompatible.


In [3]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(temperature=0.4, max_tokens=500)
